In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [6]:
load_data = pd.read_csv('/content/drive/MyDrive/FastAPI_Model_Project/insurance.csv')
print(load_data[:10])

   age  weight  height  income_lpa  smoker        city      occupation  \
0   67   119.8    1.56    2.920000   False      Jaipur         retired   
1   36   101.1    1.83   34.280000   False     Chennai      freelancer   
2   39    56.8    1.64   36.640000   False      Indore      freelancer   
3   22   109.4    1.55    3.340000    True      Mumbai         student   
4   69    62.2    1.60    3.940000    True      Indore         retired   
5   53    62.9    1.66   50.000000   False        Kota      freelancer   
6   19    80.1    1.68    3.590000    True   Hyderabad         student   
7   31   105.7    1.78   10.865821    True       Delhi  government_job   
8   73    58.0    1.58    1.780000   False  Chandigarh         retired   
9   58    74.4    1.73   43.070000   False        Pune  business_owner   

  insurance_premium_category  
0                       High  
1                        Low  
2                        Low  
3                     Medium  
4                       High  

In [7]:
print(type(load_data))

<class 'pandas.core.frame.DataFrame'>


In [8]:
# create a copy of the dataset

data = load_data.copy()
print(data.head())

   age  weight  height  income_lpa  smoker     city  occupation  \
0   67   119.8    1.56        2.92   False   Jaipur     retired   
1   36   101.1    1.83       34.28   False  Chennai  freelancer   
2   39    56.8    1.64       36.64   False   Indore  freelancer   
3   22   109.4    1.55        3.34    True   Mumbai     student   
4   69    62.2    1.60        3.94    True   Indore     retired   

  insurance_premium_category  
0                       High  
1                        Low  
2                        Low  
3                     Medium  
4                       High  


In [9]:
# applying feature engineering
# adding a new feature , bmi , bmi is created by using 2 features, weight and height.

data['bmi'] = data['weight']/(data['height']**2)


In [10]:
# now for age

def age_group(age):
  if age < 25:
    return "young"
  elif age < 40:
    return "adult"
  elif age < 60:
    return "middle_age"
  else:
    return "senior"

In [11]:
data["age_group"] = data["age"].apply(age_group)

In [12]:
# for smoker

def lifestyle_risk(row):
  if row["smoker"] and row["bmi"] > 30:
    return "high"
  elif row["smoker"] and row["bmi"] > 25:
    return "medium"
  return "low"

In [13]:
data["lifestyle_risk"] = data.apply(lifestyle_risk, axis=1)

In [14]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]

tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [15]:
# City Tier

def city_tier(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2
  return 3

In [16]:
data["city_tier"] = data["city"].apply(city_tier)

In [17]:
data.drop(columns=["age", "weight", "height", "city", "smoker"])[["bmi", "lifestyle_risk", "city_tier", "age_group", "income_lpa", "occupation"]]

,bmi,lifestyle_risk,city_tier,age_group,income_lpa,occupation
0,49.227482,low,2,senior,2.92000,retired
1,30.189017,low,1,adult,34.28000,freelancer
2,21.118382,low,2,adult,36.64000,freelancer
3,45.535900,high,1,young,3.34000,student
4,24.296875,low,2,senior,3.94000,retired
...,...,...,...,...,...,...
95,21.420747,low,2,adult,19.64000,business_owner
96,47.984483,low,1,adult,34.01000,private_job
97,18.765432,low,1,middle_age,44.86000,freelancer
98,30.521676,low,1,adult,28.30000,business_owner


In [19]:
# setting X and Y
X = data[["bmi", "lifestyle_risk", "city_tier", "age_group", "income_lpa", "occupation"]]
Y = data["insurance_premium_category"]

In [21]:
print(X.head())
# print(Y.head())

         bmi lifestyle_risk  city_tier age_group  income_lpa  occupation
0  49.227482            low          2    senior        2.92     retired
1  30.189017            low          1     adult       34.28  freelancer
2  21.118382            low          2     adult       36.64  freelancer
3  45.535900           high          1     young        3.34     student
4  24.296875            low          2    senior        3.94     retired


In [22]:
print(Y.head())

0      High
1       Low
2       Low
3    Medium
4      High
Name: insurance_premium_category, dtype: object


In [33]:
# seperating categorical and numerical features

categorical_features = ["lifestyle_risk", "city_tier", "age_group", "occupation"]
numerical_features = ["bmi", "income_lpa"]

In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

In [35]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42))
    ]
)

In [36]:
# train test split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state=1)
pipeline.fit(X_train, Y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['lifestyle_risk',
                                                   'city_tier', 'age_group',
                                                   'occupation']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [38]:
y_pred = pipeline.predict(X_test)
accuracy_score(Y_test, y_pred)

0.85

In [39]:
X_test.sample(5)

,bmi,lifestyle_risk,city_tier,age_group,income_lpa,occupation
10,22.949982,low,1,adult,32.78,business_owner
51,38.827923,high,2,middle_age,28.95,private_job
36,21.713266,low,1,senior,0.53,retired
93,23.199416,low,2,young,1.28,student
52,47.344720,low,2,young,2.96,student


In [40]:
import pickle

pickle_model_path = "model.pkl"
with open(pickle_model_path, 'wb') as file:
  pickle.dump(pipeline, file)